# 📊 Sales Report Automation — Démonstration interactive

[![GitHub](https://img.shields.io/badge/GitHub-GomuGomuNo01-181717?logo=github)](https://github.com/GomuGomuNo01/sales-report-automation)
[![Python](https://img.shields.io/badge/Python-3.9+-3776AB?logo=python&logoColor=white)](https://python.org)

---

## 🎯 Contexte métier

Une PME exporte chaque mois ses données de ventes depuis son ERP sous forme de fichiers CSV bruts.  
Un commercial passait **5 heures par mois** à :
- Fusionner les fichiers dans Excel
- Nettoyer les données (doublons, valeurs manquantes, remises mal saisies…)
- Calculer les KPIs à la main
- Créer les graphiques un par un
- Assembler le rapport pour la direction

**Ce notebook automatise l'intégralité de ce processus en quelques secondes.**

---

### Pipeline ETL + Rapport
```
CSV bruts  →  Extraction  →  Nettoyage  →  Transformation  →  Visualisation  →  PDF 6 pages
```

## ⚙️ Installation & Configuration

> **Exécutez cette cellule en premier.** Elle installe les dépendances et clone le projet.

In [ ]:
# Installation des dépendances
!pip install pandas matplotlib seaborn fpdf2 faker openpyxl -q

# Clone du projet
import os
if not os.path.exists('sales-report-automation'):
    !git clone https://github.com/GomuGomuNo01/sales-report-automation.git --quiet

os.chdir('sales-report-automation')

import sys
sys.path.insert(0, '.')

print('✅ Environnement prêt !')
print(f'   Répertoire : {os.getcwd()}')

---
## 1️⃣ Les données brutes — ce que l'ERP exporte

Chaque mois, l'ERP produit un fichier CSV par mois.  
Les données sont brutes : doublons possibles, remises en formats mixtes, statuts non harmonisés.

In [ ]:
import pandas as pd

# Génération de 12 mois de données fictives (simulation ERP)
print('Génération des données de démonstration...')
import subprocess, sys
subprocess.run([sys.executable, 'generate_data.py'], capture_output=True)

# Aperçu du fichier CSV brut
df_raw = pd.read_csv('data/raw/ventes_janvier_2024.csv', encoding='utf-8-sig')

print(f'\n📁 Fichier : ventes_janvier_2024.csv')
print(f'   {len(df_raw)} lignes  |  {df_raw.shape[1]} colonnes')
print(f'   Statuts présents : {df_raw["statut"].value_counts().to_dict()}')
print(f'   Remises : {sorted(df_raw["remise"].unique().tolist())}')
print()

df_raw.head(8).style \
    .set_caption('📄 Extrait — Données brutes CSV (janvier 2024)') \
    .set_table_styles([{'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold')]}]) \
    .background_gradient(subset=['prix_unitaire', 'quantite'], cmap='Blues')

---
## 2️⃣ Pipeline ETL — Extraction, Nettoyage, Transformation

Le pipeline charge **tous les fichiers CSV** du dossier `data/raw/` automatiquement,
quel que soit leur nombre.

In [ ]:
import logging
logging.disable(logging.CRITICAL)   # silencer les logs pour un affichage propre

from src.extractor import extract
from src.cleaner import clean
from src.transformer import transform

# ── Étape 1 : Extraction ──────────────────────────────────────
df_brut = extract()
nb_fichiers = df_brut['source_fichier'].nunique()
print(f'① Extraction    : {len(df_brut):>5} lignes  ({nb_fichiers} fichiers CSV fusionnés)')

# ── Étape 2 : Nettoyage ───────────────────────────────────────
df_propre = clean(df_brut)
print(f'② Nettoyage     : {len(df_propre):>5} lignes  (doublons, NaN, types, remises, statuts corrigés)')

# ── Étape 3 : Transformation ──────────────────────────────────
resultats = transform(df_propre)
print(f'③ Transformation: {len(resultats):>5} agréga. (KPIs, vendeurs, mois, catégories, régions, produits)')

print('\n✅ Pipeline ETL terminé avec succès')

---
## 3️⃣ KPIs — Indicateurs clés de performance

In [ ]:
kpis = resultats['kpis']

print('┌' + '─' * 47 + '┐')
print(f'│  {"RAPPORT ANNUEL — VENTES 2024":^45}│')
print('├' + '─' * 47 + '┤')
print(f'│  CA Total             : {kpis["ca_total"]:>18,.2f} €  │')
print(f'│  Commandes actives    : {kpis["nb_commandes"]:>18,}    │')
print(f'│  Panier moyen         : {kpis["panier_moyen"]:>18,.2f} €  │')
print(f'│  Taux d\'annulation   : {kpis["taux_annulation"]:>17.2f} %   │')
print('├' + '─' * 47 + '┤')
print(f'│  Vendeurs actifs      : {kpis["nb_vendeurs"]:>18}    │')
print(f'│  Références produits  : {kpis["nb_produits"]:>18}    │')
print(f'│  Remises accordées    : {kpis["remise_totale"]:>18,.2f} €  │')
print(f'│  Commandes annulées   : {kpis["nb_annulations"]:>18}    │')
print('└' + '─' * 47 + '┘')

In [ ]:
# Top vendeurs
print('🏆 Top 5 vendeurs par CA :')
top5 = resultats['par_vendeur'].head(5).copy()
top5['ca_total'] = top5['ca_total'].map('{:,.2f} €'.format)
top5.index = range(1, 6)
top5.columns = ['Vendeur', 'CA Total']
display(top5)

print('\n📅 CA par mois :')
mois = resultats['par_mois'][['mois_nom', 'ca_total']].copy()
mois.columns = ['Mois', 'CA (€)']
mois['CA (€)'] = mois['CA (€)'].map('{:,.2f}'.format)
mois.index = range(1, len(mois) + 1)
display(mois)

---
## 4️⃣ Visualisations — Les 5 graphiques générés

In [ ]:
# Génération des graphiques PNG
from src.visualizer import generate_all_charts
chemins_charts = generate_all_charts(resultats)
print('✅ 5 graphiques générés dans output/charts/')
for nom, chemin in chemins_charts.items():
    print(f'   • {chemin.split("/")[-1]}')

In [ ]:
from IPython.display import Image, display as ipy_display

print('📊 Graphique 1 — Top Vendeurs par CA')
ipy_display(Image(chemins_charts['top_vendeurs'], width=900))

In [ ]:
print('📈 Graphique 2 — Évolution Mensuelle du CA')
ipy_display(Image(chemins_charts['evolution_mensuelle'], width=900))

In [ ]:
print('🥧 Graphique 3 — Répartition par Catégorie de Produits')
ipy_display(Image(chemins_charts['repartition_categorie'], width=900))

In [ ]:
print('🗺️ Graphique 4 — Heatmap Vendeurs × Régions')
ipy_display(Image(chemins_charts['heatmap'], width=900))

In [ ]:
print('🏅 Graphique 5 — Top Produits par CA')
ipy_display(Image(chemins_charts['top_produits'], width=900))

---
## 5️⃣ Génération du rapport PDF professionnel

Tous les éléments sont assemblés dans un rapport A4 de **6 pages** :
en-tête corporate, cartes KPI, tableaux avec mise en forme, graphiques intégrés.

In [ ]:
from src.reporter import generate_report

chemin_pdf = generate_report(
    resultats=resultats,
    chemins_charts=chemins_charts,
    periode='Janvier - Décembre 2024'
)

print(f'✅ Rapport PDF généré : {chemin_pdf}')
print(f'   CA Total    : {kpis["ca_total"]:,.2f} €')
print(f'   Commandes   : {kpis["nb_commandes"]}')
print(f'   Pages       : 6 (dashboard, vendeurs, évolution, catégories, heatmap, top produits)')

# Téléchargement automatique dans Google Colab
try:
    from google.colab import files
    files.download(chemin_pdf)
    print('\n📥 Téléchargement du PDF démarré...')
except ImportError:
    print(f'\n📁 PDF disponible localement : {chemin_pdf}')

---
## ✅ Résumé

| Étape | Technologie | Résultat |
|-------|-------------|----------|
| Extraction | `pandas` | 12 CSV → 1 DataFrame unifié |
| Nettoyage | `pandas` | Doublons, NaN, types, remises, statuts corrigés |
| Transformation | `pandas` + `numpy` | CA brut/net/comptabilisé, 6 agrégations |
| Visualisation | `matplotlib` + `seaborn` | 5 graphiques PNG 150 DPI |
| Rapport | `fpdf2` | PDF A4 professionnel 6 pages |

**Gain de temps estimé : 5h → < 10 secondes par mois.**

---

📂 **[Voir le projet complet sur GitHub](https://github.com/GomuGomuNo01/sales-report-automation)**  
📄 **[Télécharger le rapport exemple](https://github.com/GomuGomuNo01/sales-report-automation/raw/main/examples/rapport_exemple_2024.pdf)**